<h1>User Experience Analysis</h1>

In [0]:
%pip install nltk gensim


In [0]:
import nltk

In [0]:
import nltk

In [0]:
nltk.download('vader_lexicon')
from nltk.sentiment import SentimentIntensityAnalyzer

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, year, lit, when, udf, explode, lower, count, size, isnan, sum, trunc
from pyspark.sql.types import IntegerType, ArrayType, StringType
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
from scipy.stats import shapiro, normaltest
from gensim import corpora
from gensim.models import LdaModel
import pandas as pd

In [0]:
spark = SparkSession.builder.appName("analysis_painpoint").getOrCreate()

In [0]:
reddit_url_to_exclude = ["https://www.reddit.com/r/korea/comments/1it9gty/exclusive_being_taken_prisoner_is_treason_in/", "https://www.reddit.com/r/korea/comments/11a53p7/colonial_police_warned_residents_about_police/", "https://www.reddit.com/r/korea/comments/13wkuv7/this_is_how_my_ukrainian_neighbor_responded_to/", "https://www.reddit.com/r/korea/comments/w6phlt/colonial_authorities_discussed_how_to_reduce/", "https://www.reddit.com/r/korea/comments/15xk99n/autopsy_identifies_strangulation_as_preliminary/", "https://www.reddit.com/r/korea/comments/6afzls/my_complicated_visa_issue_with_seemingly_no/"]

In [0]:
appstore_df = spark.table("workspace.growth_poc.silver_appstore_reviews") \
                    .filter(year("updated") >= 2023)\
                   .select(
                       col("updated").alias("review_date"),
                       col("rating").alias("review_rate"),
                       col("content_translated").alias("review_content"),
                       col("sentences").alias("review_sentences"),
                       col("words").alias("review_words"),
                       col("thumbsUpCount").alias("review_thumbsUpCount"),
                       col("appName"),
                       col("country"),
                       col("language")
                   ) 
playstore_df = spark.table("workspace.growth_poc.silver_playstore_reviews") \
                    .filter(year("at") >= 2023)\
                    .select(
                       col("at").alias("review_date"),
                       col("score").alias("review_rate"),
                       col("content_translated").alias("review_content"),
                       col("sentences").alias("review_sentences"),
                       col("words").alias("review_words"),
                       col("thumbsUpCount").alias("review_thumbsUpCount"),
                       col("appName"),
                       col("language")
                   ) 
reddit_df = spark.table("workspace.growth_poc.silver_reddit_reviews")\
                    .filter((year("created_datetime") >= 2023) & ~(col("url").isin(reddit_url_to_exclude))) \
                    .select(
                       col("created_datetime").alias("review_date"),
                       col("content_translated").alias("review_content"),
                       col("sentences").alias("review_sentences"),
                       col("words").alias("review_words"),
                       col("score").alias("review_thumbsUpCount"),
                       col("language")
                   ) 
                    
review_contents_df = appstore_df.unionByName(playstore_df,allowMissingColumns=True) \
                                .unionByName(reddit_df,allowMissingColumns=True)


In [0]:
# create temp view for faster access in later steps
review_contents_df.createOrReplaceTempView("review_contents_temp_view")
review_contents_df = spark.sql("SELECT * FROM review_contents_temp_view")

In [0]:
print(appstore_df.count())
print(playstore_df.count())
print(reddit_df.count())

<h2>Check Data</h2>

In [0]:
# 1. check schema
review_contents_df.printSchema()
print()
# 2. check column names
print(review_contents_df.columns)
print()
# 3. check data typue
print(review_contents_df.dtypes)
print()
# 4. check number of rows
print(review_contents_df.count())
print()
# 5. check statistical status
review_contents_df.describe().show()

# 6. check missing values
# get numeric columns
numeric_cols = [name for name, dtype in review_contents_df.dtypes if dtype in ('double', 'float', 'bigint')]
# numeric columns: count null values
for c in numeric_cols:
    null_count = review_contents_df.select(count(when(col(c).isNull() | isnan(c), c))).collect()[0][0]
    print(f"{c}: {null_count} nulls")



## 1. Sentiment Analysis
I will conduct sentiment analysis to find: 
<ol>
<li> If the rates given by users actually match to text sentiment
<li> Current weighted distribution between positive, neutral, and negative reviews
</ol> 

### 1-1. Users rate vs Text sentiment

In [0]:
# get review where user has given ratings
sql = """
SELECT review_rate,  
    review_content,
    review_sentences
FROM review_contents_temp_view 
WHERE review_rate IS NOT NULL AND YEAR(review_date) >= 2023
"""
review_rate_analysis_df = spark.sql(sql)



In [0]:
# 1. check schema
review_rate_analysis_df.printSchema()
print()
# 2. check column names
print(review_rate_analysis_df.columns)
print()
# 3. check data typue
print(review_rate_analysis_df.dtypes)
print()
# 4. check number of rows
print(review_rate_analysis_df.count())
print()
# 5. check statistical status
review_rate_analysis_df.describe().show()

# 6. check missing values
# get numeric columns
numeric_cols = [name for name, dtype in review_rate_analysis_df.dtypes if dtype in ('double', 'float', 'bigint')]
# numeric columns: count null values
for c in numeric_cols:
    null_count = review_rate_analysis_df.select(count(when(col(c).isNull() | isnan(c), c))).collect()[0][0]
    print(f"{c}: {null_count} nulls")



In [0]:
analyzer = SentimentIntensityAnalyzer()
def detect_text_sentiment(sentences):
  # if empty, return 0
  if not sentences:
    return 0
  # if sentence is a string, put it in an array
  if isinstance(sentences, str):
    sentences = [sentences]

  total_score = 0 
  for sentence in sentences:
    score = analyzer.polarity_scores(sentence)["compound"]
    total_score += score

  # get average of the sentiment score
  avg_score = total_score / len(sentences)

  if avg_score < 0:
    return "negative"
  elif avg_score > 0:
    return "positive"
  else: return "neutral"

detect_text_sentiment_udf = udf(detect_text_sentiment, StringType())

In [0]:
review_rate_analysis_df = review_rate_analysis_df.withColumn("classification_by_user_rate", 
                                                             when(col("review_rate") > 3, "positive")\
                                                            .when(col("review_rate") < 3, "negative")\
                                                            .otherwise("neutral"))
                                                            
rate_detected_df = review_rate_analysis_df.withColumn("classification_by_model", detect_text_sentiment_udf(col("review_sentences")))

In [0]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# convert pyspark column to list 
y_true = [row['classification_by_user_rate'] for row in rate_detected_df.select('classification_by_user_rate').collect()]
y_pred = [row['classification_by_model'] for row in rate_detected_df.select('classification_by_model').collect()]

# calculate confusion matrix
cm = confusion_matrix(y_true, y_pred)

# plot visualization
labels = sorted(list(set(y_true + y_pred)))   
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)

plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

![](./images/confusion_matrix.png)

**Findings** <br>
The VADER model performs well for detecting positive reviews, reasonably for negative reviews, but is not reliable for neutral reviews. Therefore, we should be aware that the negative reviews we analyze may not fully reflect all the negative reviews by users.
As with the word frequency analysis, I will continue to prioritize user-provided ratings to classify reviews as positive, neutral, or negative. When these ratings are not available, the VADER model will be used.


### 1-2. Current distribution of positive, neutral, and negative reviews

In [0]:
sentiment_marked_df = review_contents_df.withColumn("review_sentiment", \
        when(col("review_rate").isNull(), detect_text_sentiment_udf(col("review_sentences")))\
        .when(col("review_rate") < 3, "negative")\
        .when(col("review_rate") > 3, "positive")\
        .otherwise("neutral")   
    )
weighted_sentiment_df = sentiment_marked_df.groupBy("review_sentiment")\
                                  .agg(
                                      sum(col("review_thumbsUpCount") + 1).alias("count"))\
                                  .toPandas()

labels = weighted_sentiment_df["review_sentiment"]
sizes = weighted_sentiment_df["count"]

plt.figure(figsize=(3, 3))
plt.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90, colors=['lightcoral', 'lightblue', 'lightgreen'])
plt.title('Weighted Distribution of Reviews')
plt.axis('equal')
plt.show()

![](./images/pie_weighted_distribution.png)

**Findings**  
I analyzed the distribution of reviews based on user sentiment weighted by thumbs up count. Positive reviews account for approximately 66% of all reviews, negative reviews for about 21%, and the remaining are neutral. Comparing simple counts with weighted counts based on thumbs-up, there is no significant difference, indicating that all types of reviews receive roughly similar engagement from users.  

These results suggest that most users are satisfied with the app. However, attention should be focused on improving the experience of users who leave negative or neutral reviews, as addressing their concerns could further enhance overall user satisfaction and engagement.


##2. Word Frequency Analysis
To find the biggest pain points the users are experiencing, I conducted weighted frequency analysis.

In [0]:
keyword_analysis_df = review_contents_df.select("review_rate", "review_content", "review_words", "review_thumbsUpCount")

In [0]:
def detect_negative_review(sentence):
    if not sentence: # None or empty string
        return 0 
    score = analyzer.polarity_scores(sentence)["compound"]
    if score < 0:
        return 1
    elif score > 0:
        return -1
    else: return 0

In [0]:
# Mark negative reviews
# If review_rate is 2 or less than 2, mark as negative
# If review_rate is not available, use VADER to detect
detect_negative_review_udf = udf(detect_negative_review, IntegerType())
mark_negative_reviews_df = keyword_analysis_df.withColumn("is_negative", \
    when(col("review_rate").isNull(), detect_negative_review_udf(col("review_content")))\
    .when(col("review_rate") <= 2, 1)\
    .when(col("review_rate") > 3, -1)\
    .otherwise(0)   
)

# get only negative reviews
negative_df = mark_negative_reviews_df.filter((col("is_negative") == 1))

# flatten words
words_exploded = negative_df.select(explode(col("review_words")).alias("word"), col("review_thumbsUpCount"))

# set lowercase and count 
word_counts = words_exploded.withColumn("word", lower(col("word"))) \
                            .groupBy("word").count() \
                            .orderBy(col("count").desc()).limit(100)


In [0]:
# take a list and return a list of tuples cotaining two words
def create_bigrams_from_list(words):
    if not words or len(words) < 2:
        return []
    bigrams_list = list(zip(words[:], words[1:])) # zip stops when the short sized list meets the end
    # words[:]   = ['I',    'EAT',    'BANANA']
    # words[1:]  = ['EAT',  'BANANA']
    # => [('I', 'EAT'), ('EAT', 'BANANA')]


    bigrams = [" ".join(grams) for grams in bigrams_list]
    return bigrams

# take a list and return a list of tuples cotaining three words
def create_trigrams_from_list(words):
    if not words or len(words) < 3:
        return []
    trigrams_list = list(zip(words[:], words[1:], words[2:])) 

    trigrams = [" ".join(grams) for grams in trigrams_list]
    return trigrams

 # register the function as udf 
create_bigrams_udf = udf(create_bigrams_from_list, ArrayType(StringType()))
create_trigrams_udf = udf(create_trigrams_from_list, ArrayType(StringType()))

In [0]:
# get biagram result
bigrams_df = negative_df.withColumn("keywords_paired", 
                                    create_bigrams_udf(col("review_words"))) \
                        .select("keywords_paired", "review_thumbsUpCount")

In [0]:
# get trigram result
trigrams_df = negative_df.withColumn("keywords_paired", 
                                    create_trigrams_udf(col("review_words"))) \
                         .select("keywords_paired", "review_thumbsUpCount")

###2-1. Single Keyword Frequency Analysis

In [0]:
weighted_word_counts = words_exploded.withColumn("word", lower(col("word"))) \
    .groupBy("word") \
    .agg(
        sum(col("review_thumbsUpCount") + 1).alias("weighted_count")) \
    .orderBy(col("weighted_count").desc()) \
    .limit(100)


In [0]:
weighted_word_counts.show(100, truncate = False)

| word        | weighted_count |
|-------------|----------------|
| delivery    | 2201           |
| food        | 1797           |
| order       | 1506           |
| korea       | 1484           |
| korean      | 1474           |
| arc         | 1338           |
| phone       | 1277           |
| get         | 1240           |
| card        | 1200           |
| number      | 1198           |
| even        | 1118           |
| like        | 1062           |
| apps        | 1036           |
| app         | 1017           |
| time        | 929            |
| account     | 928            |
| restaurant  | 925            |
| one         | 854            |
| foreigner   | 813            |
| people      | 808            |
| thing       | 802            |
| review      | 786            |
| need        | 782            |
| back        | 763            |
| country     | 695            |
| go          | 668            |
| place       | 614            |
| use         | 602            |
| item        | 589            |
| way         | 588            |
| got         | 572            |
| know        | 562            |
| pay         | 556            |
| bank        | 555            |
| make        | 538            |
| money       | 510            |
| also        | 509            |
| find        | 504            |
| bad         | 499            |
| really      | 495            |
| service     | 474            |
| every       | 470            |
| work        | 466            |
| want        | 465            |
| connected   | 442            |
| still       | 436            |
| deal        | 429            |
| price       | 425            |
| shuttle     | 412            |
| sub         | 403            |
| registration| 400            |
| business    | 396            |
| register    | 389            |
| driver      | 389            |
| wanted      | 366            |
| tried       | 363            |
| visa        | 363            |
| city        | 362            |
| good        | 361            |
| visit       | 360            |
| without     | 357            |
| problem     | 354            |
| alien       | 350            |
| everything  | 348            |
| think       | 347            |
| day         | 346            |
| identity    | 343            |
| verify      | 339            |
| longterm    | 336            |
| wont        | 335            |
| week        | 335            |
| call        | 330            |
| thought     | 328            |
| come        | 317            |
| let         | 315            |
| many        | 314            |
| used        | 313            |
| isnt        | 310            |
| much        | 310            |
| plastic     | 307            |
| usually     | 306            |
| tip         | 305            |
| able        | 303            |
| great       | 300            |
| another     | 295            |
| spend       | 294            |
| see         | 290            |
| system      | 290            |
| tourist     | 284            |
| feel        | 284            |
| though      | 282            |
| might       | 282            |
| never       | 279            |
| may         | 278            |
| going       | 277            |
| two         | 275            |
| person      | 274            |
| first       | 274            |
| living      | 274            |
| big         | 274            |



###2-2. Bigram Analysis

In [0]:
weighted_bigrams_flat_df = bigrams_df.withColumn("keywords", explode(col("keywords_paired")))\
                            .groupBy("keywords")\
                            .agg(
                                sum(col("review_thumbsUpCount") + 1).alias("weighted_count")) \
                            .orderBy(col("weighted_count").desc()).limit(100)


In [0]:
weighted_bigrams_flat_df.show(50, truncate = False)

| keywords          | weighted_count |
|------------------|----------------|
| phone number      | 906           |
| food delivery     | 509           |
| need phone        | 348           |
| visit sub         | 344           |
| alien registration| 343           |
| arc alien         | 343           |
| registration card | 343           |
| number connected  | 340           |
| connected arc     | 340           |
| visa arc          | 338           |
| longterm visa     | 336           |
| apps longterm     | 336           |
| identity apps     | 336           |
| arc arc           | 336           |
| register verify   | 336           |
| arc food          | 336           |
| order register    | 336           |
| verify identity   | 336           |
| deal need         | 336           |
| delivery account  | 336           |



###2-3. Trigram Analysis

In [0]:
weighted_trigrams_flat_df = trigrams_df.withColumn("keywords", explode(col("keywords_paired")))\
                            .groupBy("keywords")\
                            .agg(
                                sum(col("review_thumbsUpCount") + 1).alias("weighted_count")) \
                            .orderBy(col("weighted_count").desc()).limit(100)

In [0]:
weighted_trigrams_flat_df.show(20, truncate = False)

| keywords                | weighted_count |
|-------------------------|----------------|
| need phone number       | 348           |
| alien registration card | 343           |
| arc alien registration  | 343           |
| connected arc alien     | 340           |
| number connected arc    | 340           |
| phone number connected  | 340           |
| arc arc food            | 336           |
| identity apps longterm  | 336           |
| deal need phone         | 336           |
| order register verify   | 336           |
| longterm visa arc       | 336           |
| verify identity apps    | 336           |
| registration card order | 336           |
| food delivery account   | 336           |
| visa arc arc            | 336           |
| card order register     | 336           |
| arc food delivery       | 336           |
| register verify identity| 336           |
| apps longterm visa      | 336           |
| korean phone number     | 299           |



By applying weighted keyword analysis using thumbs-up counts, we were able to identify which issues users are most strongly resonating with, beyond just frequency of mentions.

<ol>
<li><b>Dominant Issue: Verification & Payment</b></li>
Key Words:
<ul>
<li>N-grams: phone number (906), need phone (348), alien registration (343), registration card (343), number connected (340), visa arc (338), longterm visa (336), register verify (336), verify identity (336), delivery account (336), need phone number (348), alien registration card (343), arc alien registration (343), connected arc alien (340), number connected arc (340), phone number connected (340), order register verify (336), longterm visa arc (336), verify identity apps (336), register verify identity (336), korean phone number (299)</li>
<li>Single Keywords: korean (1474), arc (1338), phone (1277), card (1200), number (1198), account (928), bank (555), money (510), registration (400), identity (343), verify (339)</li>
</ul>
Quantitative evidence shows that verification and payment issues are the top concerns. In particular, challenges related to ARC (Alien Registration Card), identity verification, and using Korean bank accounts or cards are confirmed as the problems that users most strongly resonate with.

<li><b>Service Usage & Ordering Experience</b></li>
Key Words:
<ul>
<li>Single Keywords: time (929), item (589), service (474), shuttle (412), sub (403), driver (389)</li>
</ul>
Unlike simple frequency analysis, issues related to service usage and the ordering process do not appear as top concerns in the weighted analysis.

<li><b>Foreigner-Specific Issues</b></li>
Key Words:
<ul>
<li>Single Keywords: korean (1474), foreigner (813), tourist (284)</li>
</ul>
Issues specific to foreign users also do not carry as much weight when thumbs-up counts are applied. Interpretation may vary depending on whether "Korean" is read as language support or as referring to Korean-specific systems/items.

<li><b>Notable Points</b></li>
Key Words:
<ul>
<li>Single Keywords: shuttle (412), alternative left shuttle (200), plastic (307)</li>
</ul>
Unlike the single keyword analysis, mentions of Coupang Eats do not appear. Instead, Shuttle emerges as an alternative delivery solution. The appearance of "plastic" reflects user concern about plastic waste generated through food delivery apps.
</ol>


###2-4. LDA

In [0]:
review_contents_df.printSchema()

lda_start_df = review_contents_df.withColumn("review_sentiment", \
    when(col("review_rate").isNull(), detect_text_sentiment_udf(col("review_sentences")))\
    .when(col("review_rate") < 3, "negative") \
    .when(col("review_rate") > 3, "positive")\
    .otherwise("neutral")
)

# Assuming neutral reviews contain opinions about improvements,
# I will combine netural and negative reviews.
lda_start_df = lda_start_df.withColumn("review_sentiment",
                                       when(col("review_sentiment") == "neutral", "negative").otherwise(col("review_sentiment")))


In [0]:
lda_negative_df = lda_start_df.filter(col("review_sentiment") == "negative")
lda_negative_collect = lda_negative_df.select("review_words", "review_thumbsUpCount").collect()
negative_reviews_words = [row["review_words"] for row in lda_negative_collect]
negative_reviews_weight = [row["review_thumbsUpCount"]+1 for row in lda_negative_collect]

negative_reviews = []
for words, weight in zip(negative_reviews_words, negative_reviews_weight):
    for _ in range(weight):
        negative_reviews.append(words)

# take unique words from the nested list and map with auto-generated id
dictionary = corpora.Dictionary(negative_reviews)

# get word frequency for each review, using dictionary
corpus = [dictionary.doc2bow(doc) for doc in negative_reviews]
 


In [0]:
# train LDA model
# 9, 11, 12, // 여기까지 그냥저냥
# 32 여기부터 trash 발견, stock, 요기요 등장, overwork, underpaid
# 36도 괜찮은데 # foreigner understand 나옴
# 23 good

NUM_TOPICS = 23 # start with 3 topics
ldamodel = LdaModel(corpus, num_topics=NUM_TOPICS, id2word=dictionary, passes=15, random_state=42)

# check result
topics = ldamodel.print_topics(num_words=5, num_topics=NUM_TOPICS) # print top 5 words from each topic
print("\nLDA topic modelling result:")
for topic in topics:
    print(topic)


# # 특정 문서의 토픽 분포 확인
# doc_num = 0
# doc_bow = corpus[doc_num]
# doc_topics = ldamodel.get_document_topics(doc_bow)
# print(f"\n문서 '{sample_data[doc_num]}'의 토픽 분포:")
# for topic_num, prop in doc_topics:
#     print(f"  토픽 #{topic_num}: {prop:.4f}")

In [0]:
lda_app_negative_df = lda_negative_df.filter(col("appName").isNotNull()).select(["review_words", "review_thumbsUpCount"])

lda_app_negative_collect = lda_app_negative_df.collect()
negative_app_words = [row["review_words"] for row in lda_app_negative_collect]
negative_app_count =  [row["review_thumbsUpCount"]+1 for row in lda_app_negative_collect]

negative_app_reviews = []

for words, count in zip(negative_app_words, negative_app_count):
    for _ in range(count):
        negative_app_reviews.append(words)
    


In [0]:
# take unique words from the nested list and map with auto-generated id
dictionary = corpora.Dictionary(negative_app_reviews)

# get word frequency for each review, using dictionary
corpus = [dictionary.doc2bow(doc) for doc in negative_app_reviews]


In [0]:
from gensim.models.coherencemodel import CoherenceModel

coherence_values = []
# K값의 범위를 2부터 10까지 테스트
for i in range(2, 40):
    ldamodel = LdaModel(corpus, num_topics=i, id2word=dictionary, random_state=42, passes = 15)
    coherence_model_lda = CoherenceModel(model=ldamodel, texts=negative_app_reviews, dictionary=dictionary, coherence='c_v')
    coherence_lda = coherence_model_lda.get_coherence()
    coherence_values.append(coherence_lda)

# 결과 시각화
x = range(2, 40)
plt.plot(x, coherence_values)
plt.xlabel("Number of Topics (K)")
plt.ylabel("Coherence score")
plt.title("Finding Optimal K using Coherence Score")
plt.grid(True)
plt.show()

In [0]:
# train LDA model
# 16 good but I want more granularity
# 20: translate is gone
# 28: perfect but too good?
# 14 good. transla어
NUM_TOPICS = 13 # start with 3 topics
ldamodel = LdaModel(corpus, num_topics=NUM_TOPICS, id2word=dictionary, passes=10, random_state=42)

# check result
topics = ldamodel.print_topics(num_words=5, num_topics=NUM_TOPICS) # print top 5 words from each topic
print("\nLDA topic modelling result:")
for topic in topics:
    print(topic)


# # 특정 문서의 토픽 분포 확인
# doc_num = 0
# doc_bow = corpus[doc_num]
# doc_topics = ldamodel.get_document_topics(doc_bow)
# print(f"\n문서 '{sample_data[doc_num]}'의 토픽 분포:")
# for topic_num, prop in doc_topics:
#     print(f"  토픽 #{topic_num}: {prop:.4f}")

In [0]:
print(len(negative_app_words))
print(len(negative_app_reviews))

In [0]:
lda_negative_df = lda_negative_df.withColumn("year_month", trunc("review_date", "month"))

In [0]:
def get_topic_for_review(review_words):

    if not review_words:
        return None
    new_corpus = dictionary.doc2bow(review_words)
    
    topic_distribution = ldamodel.get_document_topics(new_corpus)

    if not topic_distribution:
        return -1 

    dominant_topic = sorted(topic_distribution, key=lambda x: x[1], reverse=True)[0]
    topic_num = dominant_topic[0]
 

    return int(topic_num) 

# UDF 등록
get_topic_udf = udf(get_topic_for_review, IntegerType())



In [0]:
lda_coupang_eats_negative_df = lda_negative_df.filter(col("appName")=="coupangEats").select(["review_words", "review_thumbsUpCount", "year_month"])

lda_coupang_eats_negative_collect = lda_coupang_eats_negative_df.collect()
negative_coupang_eats_words = [row["review_words"] for row in lda_coupang_eats_negative_collect]
negative_coupang_eats_count =  [row["review_thumbsUpCount"]+1 for row in lda_coupang_eats_negative_collect]


negative_coupang_eats_reviews = []


for words, count, date in zip(negative_coupang_eats_words, negative_coupang_eats_count):
    for _ in range(count):
        negative_coupang_eats_reviews.append(words)
    
# take unique words from the nested list and map with auto-generated id
dictionary = corpora.Dictionary(negative_coupang_eats_reviews)

# get word frequency for each review, using dictionary
corpus = [dictionary.doc2bow(doc) for doc in negative_coupang_eats_reviews]

# train LDA model
# 7, 14
NUM_TOPICS = 7
ldamodel = LdaModel(corpus, num_topics=NUM_TOPICS, id2word=dictionary, passes=10, random_state=42)


# check result
topics = ldamodel.print_topics(num_words=5, num_topics=NUM_TOPICS) # print top 5 words from each topic
print("\nLDA topic modelling result:")
for topic in topics:
    print(topic)




In [0]:
from gensim.models.phrases import Phrases, Phraser
phrases = Phrases(negative_coupang_eats_reviews, min_count=1, threshold=3)
bigram = Phraser(phrases)
bigram_reviews = [bigram[doc] for doc in negative_coupang_eats_reviews]

# take unique words from the nested list and map with auto-generated id
dictionary = corpora.Dictionary(bigram_reviews)

# get word frequency for each review, using dictionary
corpus = [dictionary.doc2bow(doc) for doc in bigram_reviews]

# train LDA model
# 7, 14
NUM_TOPICS = 7
ldamodel = LdaModel(corpus, num_topics=NUM_TOPICS, id2word=dictionary, passes=10, random_state=42)


# check result
topics = ldamodel.print_topics(num_words=5, num_topics=NUM_TOPICS) # print top 5 words from each topic
print("\nLDA topic modelling result:")
for topic in topics:
    print(topic)

In [0]:
lda_coupang_with_topic = lda_coupang_eats_negative_df.withColumn("topic", get_topic_udf(col("review_words"))).toPandas()

coupang_conditions = [
    lda_coupang_with_topic['topic'].isin([0,2,3]),
    lda_coupang_with_topic['topic'].isin([1]),
    lda_coupang_with_topic['topic'].isin([6]),
    lda_coupang_with_topic['topic'].isin([4,5])
]

coupang_labels = (['payment', 'registration', 'language', 'app experience'])

lda_coupang_with_topic['label'] = np.select(coupang_conditions, coupang_labels, default = 'etc')
lda_coupang_with_topic['count'] = lda_coupang_with_topic.groupby(['year_month', 'label'])['label'].transform('size')


lda_coupang_summary = (
    lda_coupang_with_topic
    .groupby(['year_month', 'label'])
    .size()
    .reset_index(name='count')  
)

monthly_total = (
    lda_coupang_summary
    .groupby('year_month')['count']
    .sum()
    .reset_index(name='monthly_total')
)

lda_coupang = pd.merge(lda_coupang_summary, monthly_total, on='year_month')
lda_coupang['ratio'] = lda_coupang['count'] / lda_coupang['monthly_total']



In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df_pivot = df.pivot_table(
    index='year_month', 
    columns='label', 
    values='ratio'
)

# 2. NaN 값을 0으로 채우기 (Fill NaN with 0)
df_filled = df_pivot.fillna(0)

# 3. 그래프를 그리기 위해 다시 긴 형식(Long Format)으로 변환
df_final = df_filled.reset_index().melt(
    id_vars='year_month', 
    var_name='label', 
    value_name='ratio'
)

# 4. 데이터 정렬
df_final = df_final.sort_values(by='year_month')
# 분기 및 연도 컬럼 생성
df['year'] = df['year_month'].dt.year
df['quarter'] = df['year_month'].dt.quarter

# 분기 레이블 (예: 2023-Q1) 생성
df['quarter_label'] = df['year'].astype(str) + '-Q' + df['quarter'].astype(str)

df_quarterly = df.groupby(['year', 'quarter', 'quarter_label', 'label']).agg(
    quarterly_count=('count', 'sum'),
    quarterly_total=('monthly_total', 'sum')
).reset_index()

# 분기별 최종 비율 재계산
df_quarterly['ratio'] = df_quarterly['quarterly_count'] / df_quarterly['quarterly_total']

# -------------------------------------------------------------
# 라인 그래프 생성 (텍스트 영어로 설정)
# -------------------------------------------------------------

plt.figure(figsize=(12, 7))

sns.lineplot(
    data=df_quarterly,  # 분기별 집계 데이터 사용
    x='quarter_label',  # X축은 분기 레이블
    y='ratio', 
    hue='label', 
    marker='o',
    linewidth=2
)

# 5. 그래프 설정 및 레이블 (영문)
plt.title('Quarterly Trend of Negative Review Topic Ratios', fontsize=16, pad=20)
plt.xlabel('Year-Quarter', fontsize=12)
plt.ylabel('Proportion of Total Quarterly Negative Reviews', fontsize=12)

# Y축 백분율 형식
plt.gca().yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))

# X축 레이블 회전
plt.xticks(rotation=45, ha='right')

# 범례 설정
plt.legend(title='Negative Topic', loc='best')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()

plt.show()

In [0]:
lda_baemin_negative_df = lda_negative_df.filter(col("appName")=="baemin").select(["review_words", "review_thumbsUpCount", "year_month"])

lda_baemin_negative_collect = lda_baemin_negative_df.collect()
negative_baemin_words = [row["review_words"] for row in lda_baemin_negative_collect]
negative_baemin_count =  [row["review_thumbsUpCount"]+1 for row in lda_baemin_negative_collect]

negative_baemin_reviews = []

for words, count in zip(negative_baemin_words, negative_baemin_count):
    for _ in range(count):
        negative_baemin_reviews.append(words)
    
# take unique words from the nested list and map with auto-generated id
dictionary = corpora.Dictionary(negative_baemin_reviews)

# get word frequency for each review, using dictionary
corpus = [dictionary.doc2bow(doc) for doc in negative_baemin_reviews]

# train LDA model
# 4,6,8,9
NUM_TOPICS = 7
ldamodel = LdaModel(corpus, num_topics=NUM_TOPICS, id2word=dictionary, passes=10, random_state=42)

# check result
topics = ldamodel.print_topics(num_words=5, num_topics=NUM_TOPICS) # print top 5 words from each topic
print("\nLDA topic modelling result:")
for topic in topics:
    print(topic)

# 전체 코퍼스에서 토픽별 분포 계산
# topic_weights = [max(ldamodel[doc], key=lambda x: x[1])[0] for doc in corpus]
# from collections import Counter
# topic_freq = Counter(topic_weights)
# print(topic_freq)
 


In [0]:
phrases = Phrases(negative_baemin_reviews, min_count=1, threshold=2)
bigram = Phraser(phrases)
bigram_reviews = [bigram[doc] for doc in negative_baemin_reviews]

# take unique words from the nested list and map with auto-generated id
dictionary = corpora.Dictionary(bigram_reviews)

# get word frequency for each review, using dictionary
corpus = [dictionary.doc2bow(doc) for doc in bigram_reviews]

# train LDA model
# 7, 14
NUM_TOPICS = 7
ldamodel = LdaModel(corpus, num_topics=NUM_TOPICS, id2word=dictionary, passes=10, random_state=42)


# check result
topics = ldamodel.print_topics(num_words=5, num_topics=NUM_TOPICS) # print top 5 words from each topic
print("\nLDA topic modelling result:")
for topic in topics:
    print(topic)

In [0]:

lda_baemin_with_topic = lda_baemin_negative_df.withColumn("topic", get_topic_udf(col("review_words"))).toPandas()

baemin_conditions = [
    lda_baemin_with_topic['topic'].isin([0,2,3]),
    lda_baemin_with_topic['topic'].isin([1]),
    lda_baemin_with_topic['topic'].isin([6]),
    lda_baemin_with_topic['topic'].isin([4,5])
]

baemin_labels = (['payment','registration', 'language','app experience'])
df_baemin_labeled = lda_baemin_with_topic.copy()
df_baemin_labeled['label'] = np.select(baemin_conditions, baemin_labels, default='etc')

df_baemin_summary = (
    df_baemin_labeled
    .groupby(['year_month', 'label'])
    .size()
    .reset_index(name='count')
)

df_baemin_monthly_total = (
    df_baemin_summary
    .groupby('year_month')['count']
    .sum()
    .reset_index(name='monthly_total')
)

all_months = df_baemin_monthly_total['year_month'].unique()
all_labels = df_baemin_summary['label'].unique()

df_baemin_full_key = pd.MultiIndex.from_product([all_months, all_labels], names=['year_month', 'label']).to_frame(index=False)

df_baemin_full = pd.merge(
    df_baemin_full_key,
    df_baemin_summary,
    on=['year_month', 'label'],
    how='left'
)

df_baemin_full['count'] = df_baemin_full['count'].fillna(0)

df_baemin_full = pd.merge(df_baemin_full, df_baemin_monthly_total, on='year_month', how='left')

df_baemin_full['ratio'] = np.where(
    df_baemin_full['monthly_total'] == 0,
    0,
    df_baemin_full['count'] / df_baemin_full['monthly_total']
)

df_baemin_full['year_month'] = pd.to_datetime(df_baemin_full['year_month'])

df_baemin_full['year'] = df_baemin_full['year_month'].dt.year
df_baemin_full['quarter'] = df_baemin_full['year_month'].dt.quarter

df_baemin_full['quarter_label'] = df_baemin_full['year'].astype(str) + '-Q' + df_baemin_full['quarter'].astype(str)

df_baemin_quarterly = df_baemin_full.groupby(['year', 'quarter', 'quarter_label', 'label']).agg(
    quarterly_count=('count', 'sum'),
    quarterly_total=('monthly_total', 'sum')
).reset_index()

df_baemin_quarterly['ratio'] = df_baemin_quarterly['quarterly_count'] / df_baemin_quarterly['quarterly_total']

plt.figure(figsize=(12, 7))

sns.lineplot(
    data=df_baemin_quarterly,
    x='quarter_label',
    y='ratio',
    hue='label',
    marker='o',
    linewidth=2
)

plt.title('Quarterly Trend of Baemin Negative Review Topic Ratios', fontsize=16, pad=20)
plt.xlabel('Year-Quarter', fontsize=12)
plt.ylabel('Proportion of Total Quarterly Negative Reviews', fontsize=12)

plt.gca().yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))

plt.xticks(rotation=45, ha='right')

plt.legend(title='Negative Topic', loc='best')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()

plt.show()
 

In [0]:
lda_yogiyo_negative_df = lda_negative_df.filter(col("appName")=="yogiyo").select(["review_words", "review_thumbsUpCount", "year_month"])

lda_yogiyo_negative_collect = lda_yogiyo_negative_df.collect()
negative_yogiyo_words = [row["review_words"] for row in lda_yogiyo_negative_collect]
negative_yogiyo_count =  [row["review_thumbsUpCount"]+1 for row in lda_yogiyo_negative_collect]

negative_yogiyo_reviews = []

for words, count in zip(negative_yogiyo_words, negative_yogiyo_count):
    for _ in range(count):
        negative_yogiyo_reviews.append(words)
    
# take unique words from the nested list and map with auto-generated id
dictionary = corpora.Dictionary(negative_yogiyo_reviews)

# get word frequency for each review, using dictionary
corpus = [dictionary.doc2bow(doc) for doc in negative_yogiyo_reviews]

# train LDA model
# 4, 7, 10, 15
NUM_TOPICS = 6  
ldamodel = LdaModel(corpus, num_topics=NUM_TOPICS, id2word=dictionary, passes=10, random_state=42)

# check result
topics = ldamodel.print_topics(num_words=5, num_topics=NUM_TOPICS) # print top 5 words from each topic
print("\nLDA topic modelling result:")
for topic in topics:
    print(topic)




In [0]:
phrases = Phrases(negative_yogiyo_reviews, min_count=1, threshold=2)
bigram = Phraser(phrases)
bigram_reviews = [bigram[doc] for doc in negative_yogiyo_reviews]

# take unique words from the nested list and map with auto-generated id
dictionary = corpora.Dictionary(bigram_reviews)

# get word frequency for each review, using dictionary
corpus = [dictionary.doc2bow(doc) for doc in bigram_reviews]

# train LDA model
# 7, 14
NUM_TOPICS = 7
ldamodel = LdaModel(corpus, num_topics=NUM_TOPICS, id2word=dictionary, passes=10, random_state=42)


# check result
topics = ldamodel.print_topics(num_words=5, num_topics=NUM_TOPICS) # print top 5 words from each topic
print("\nLDA topic modelling result:")
for topic in topics:
    print(topic)

In [0]:
lda_yogiyo_with_topic = lda_yogiyo_negative_df.withColumn("topic", get_topic_udf(col("review_words"))).toPandas()

yogiyo_conditions = [
    lda_yogiyo_with_topic['topic'].isin([2,6]),
    lda_yogiyo_with_topic['topic'].isin([3]),
    lda_yogiyo_with_topic['topic'].isin([1]),
    lda_yogiyo_with_topic['topic'].isin([0,4,5])
]
yogiyo_labels = ['payment', 'registration', 'language', 'app experience']

df_yogiyo_labeled = lda_yogiyo_with_topic.copy()
df_yogiyo_labeled['label'] = np.select(yogiyo_conditions, yogiyo_labels, default='etc')

df_yogiyo_summary = (
    df_yogiyo_labeled
    .groupby(['year_month', 'label'])
    .size()
    .reset_index(name='count')
)

df_yogiyo_monthly_total = (
    df_yogiyo_summary
    .groupby('year_month')['count']
    .sum()
    .reset_index(name='monthly_total')
)

all_months_y = df_yogiyo_monthly_total['year_month'].unique()
all_labels_y = df_yogiyo_summary['label'].unique()

df_yogiyo_full_key = pd.MultiIndex.from_product([all_months_y, all_labels_y], names=['year_month', 'label']).to_frame(index=False)

df_yogiyo_full = pd.merge(
    df_yogiyo_full_key,
    df_yogiyo_summary,
    on=['year_month', 'label'],
    how='left'
)

df_yogiyo_full['count'] = df_yogiyo_full['count'].fillna(0)

df_yogiyo_full = pd.merge(df_yogiyo_full, df_yogiyo_monthly_total, on='year_month', how='left')

df_yogiyo_full['ratio'] = np.where(
    df_yogiyo_full['monthly_total'] == 0,
    0,
    df_yogiyo_full['count'] / df_yogiyo_full['monthly_total']
)

df_yogiyo_full['year_month'] = pd.to_datetime(df_yogiyo_full['year_month'])

df_yogiyo_full['year'] = df_yogiyo_full['year_month'].dt.year
df_yogiyo_full['quarter'] = df_yogiyo_full['year_month'].dt.quarter

df_yogiyo_full['quarter_label'] = df_yogiyo_full['year'].astype(str) + '-Q' + df_yogiyo_full['quarter'].astype(str)

df_yogiyo_quarterly = df_yogiyo_full.groupby(['year', 'quarter', 'quarter_label', 'label']).agg(
    quarterly_count=('count', 'sum'),
    quarterly_total=('monthly_total', 'sum')
).reset_index()

df_yogiyo_quarterly['ratio'] = df_yogiyo_quarterly['quarterly_count'] / df_yogiyo_quarterly['quarterly_total']

plt.figure(figsize=(12, 7))

sns.lineplot(
    data=df_yogiyo_quarterly,
    x='quarter_label',
    y='ratio',
    hue='label',
    marker='o',
    linewidth=2
)

plt.title('Quarterly Trend of Yogiyo Negative Review Topic Ratios', fontsize=16, pad=20)
plt.xlabel('Year-Quarter', fontsize=12)
plt.ylabel('Proportion of Total Quarterly Negative Reviews', fontsize=12)

plt.gca().yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))

plt.xticks(rotation=45, ha='right')

plt.legend(title='Negative Topic', loc='best')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()

plt.show()


In-depth User Review Analysis via LDA Topic Modeling
1. Initial Integrated Analysis (First Iteration)
Objective: To conduct a broad exploration of key pain points for foreign users by integrating reviews from diverse channels, including Reddit and app stores.

Methodology:

The optimal number of topics (K) was set to 23, based on a combination of Coherence Scores and qualitative judgment.

Review data was weighted by the number of upvotes to reflect significance.

Key Findings:

Payment/Authentication Barriers: Topics such as [“bank”, “card”, “account”, “use”, “korean”] and [“arc”, “visa”, “register”] clearly emerged, highlighting the difficulties associated with foreigner identification and using the Korean payment system.

Order/Delivery Process Issues: Various friction points were identified, including [“item”, “order”, “stock”] (out-of-stock items), [“try”, “figure”, “deliver”, “address”] (address input errors), and [“delivery, trash, food, plastic”] (concerns about excessive packaging waste).

Limitations & Rationale for Second Iteration:

It was determined that consolidating data from multiple sources introduced "noise," which reduced the overall clarity and distinctiveness of the topics.

To derive more actionable insights, it became necessary to focus on data directly related to the actual app experience.

2. Focused App Store Review Analysis (Second Iteration)
Objective: To overcome the limitations of the first analysis and generate clear, actionable recommendations by focusing on direct feedback from actual app users.

Methodology:

The scope of the analysis was narrowed to App Store and Play Store reviews (153 raw, 523 after weighting).

Based on this refined dataset, the optimal number of topics (K) was recalibrated to 13.

Core Findings:

The analysis revealed that user pain points converged into three primary categories: 1) Payment/Authentication, 2) Language Support, and 3) Delivery Service.

Key Insight: Notably, topics related to "Delivery Service" (e.g., food being left at the door) suggest an issue deeper than the delivery experience itself. It strongly implies a "communication failure with delivery drivers" stemming from the language barrier. This indicates that enhancing language support could directly impact delivery satisfaction.

3. Conclusion & Proposal for Future Analysis
A comparative analysis across major apps (Coupang Eats, Baemin, Yogiyo) confirmed that these topics related to payment, language, and delivery are universal problems within the market for foreign users.

For future work, a time-series analysis comparing major app update dates with the frequency of specific keywords (e.g., 'card', 'address') is recommended. This would serve as a key performance indicator to measure the effectiveness of feature improvements in resolving user-identified issues.

## 3. Trend Analysis
Now, I will look into the trend of non-positive reviews over time by:
<ol>
<li> Calculate non-positive review ratio over time
<li> Plot trend line and reference baselines
</ol> 

### 3-1. Non positive review ratio trend over time
I will plot line graphs to visualize the trend of ratio of non-positive reviews over time. Since we observed no significant difference between simple and weighted counts in the overall review distribution, only simple counts will be used in this graph.
To the graph, I would like to add a baseline to evaluate the trend.  
To determine an appropriate baseline, I first need to check whether the data follows a normal distribution.  
If the data is approximately normal, I will set the baseline using Mean + Standard Deviation as the upper bound and Mean - Standard Deviation as the lower bound.  
If the data is not normal, I will instead use the third quartile (Q3) as the baseline.  


In [0]:
from pyspark.sql.functions import to_date
sentiment_date_df = sentiment_marked_df.withColumn("review_date", to_date(col("review_date")))

In [0]:
sentiment_date_df.printSchema()

In [0]:
from pyspark.sql.functions import count
sentiment_date_agg_df = sentiment_date_df.groupBy("review_date", "review_sentiment").agg(
    count("*").alias("count"),
    sum(col("review_thumbsUpCount") + 1).alias("weighted_count")
)

In [0]:
# the date is at day level -> aggregate to year-month level
# truc, month : return the first day of the given month
monthly_df = sentiment_date_agg_df.withColumn("year_month", trunc("review_date", "month"))

# grouping
sentiment_trend_df = monthly_df.groupBy("year_month") \
    .pivot("review_sentiment", ["positive", "negative", "neutral"]) \
    .agg(sum("count")).na.fill(0)\
    .orderBy("year_month")

pd_df = sentiment_trend_df.toPandas()

In [0]:
# add negative and netural reviews
pd_df['non_positive'] = pd_df['negative'] + pd_df['neutral']
pd_df['all_reviews'] = pd_df['positive'] + pd_df['non_positive'] 
# if there is any review, calcaulte the ratio
# otherwise, populate np.nan (not 0, to prevent 'divided by zero')
pd_df['non_positive_ratio'] = np.where(
    pd_df['all_reviews'] > 0, pd_df['non_positive']/pd_df['all_reviews'],
    np.nan
)

# drop rows with na in ratio
plot_df = pd_df.dropna(subset=['non_positive_ratio']).copy()

In [0]:
data = plot_df["non_positive_ratio"]

# Check Skewness, Kurtosis
print("Skewness:", data.skew())     
print("Kurtosis:", data.kurt())     

# Shapiro-Wilk Test
stat, p = shapiro(data)
print(f"Shapiro-Wilk Test: stat={round(stat,2)}, p={round(p,2)}")

# D’Agostino and Pearson’s Test (scipy.stats.normaltest)
stat, p = normaltest(data)
print(f"D’Agostino and Pearson’s Test: stat={round(stat,2)}, p={round(p,2)}")

Skewness: close to 0 => symmetrical <br>
Kurtosis: a bit flat <br>
Shapiro-Wilk Test: p > 0.05 => normal distribution <br>
D Agostino and Pearson's Test: p > 0.85 => normal distrbution <br>

Therefore, I will use Mean + Standard Deviation as upper bound and Mean - Standard Deviation as the lower bound.


In [0]:
# convert date to numeric value to calculate trend line (linear regression)
x_numeric = mdates.date2num(plot_df['year_month'])
y_values = plot_df['non_positive_ratio']

# find slope and intercept
slope, intercept = np.polyfit(x_numeric, y_values, deg = 1) # deg = 1 : linear
# using the slope and intercept, calculate reference line
trend_y = slope * x_numeric + intercept

# calculate mean and std to set base line
ratio_mean = y_values.mean()
ratio_std = y_values.std()

upper_bound = ratio_mean + ratio_std
lower_bound = ratio_mean - ratio_std

# plot graph
fig, ax = plt.subplots(figsize = (15,7))
ax.plot(plot_df['year_month'], y_values, marker = 'o', linestyle = '-', label = "Monthly Ratio", color = "green")

# add reference line
ax.plot(plot_df['year_month'], trend_y, linestyle='--', linewidth= 1.2, label = "Reference line", color = "red")

# add base line
ax.axhline(y=upper_bound, color='orange', linestyle='--', linewidth=1.2, label='Upper Bound')
ax.axhline(y=lower_bound, color='orange', linestyle='--', linewidth=1.2, label='Lower Bound')

ax.set_title('Non-Positive Ratio Over Time', fontsize=16)
ax.set_xlabel('Month')
ax.set_ylabel('Ratio (Non-Positive / All Reviews)')
ax.legend()
ax.grid(True, which='both', linestyle='--', linewidth=0.5)
ax.set_ylim(bottom=0)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

**Findings**  
The green line shows the ratio of non-positive reviews, the yellow dashed lines represent the upper and lower bounds, and the red dashed line indicates the trend line.
Based on the linear regression analysis, the long-term trend line (red) for the non-positive review ratio remains almost flat with no meaningful slope. This suggests that over the past three years, the overall user experience has been consistent, staying around 34%.

To improve user experience, we should pay attention to the short-term fluctuations that can move this stagnant trend. In May–July 2023, September 2023, and June 2024, the non-positive review ratio rose sharply above the upper bound. This indicates that there were serious issues that caused user inconvenience. Analyzing the main complaints in the reviews from these periods could help identify the most urgent problems to address.

On the other hand, in February, August, and December 2023, as well as September–October 2024 and March 2025, the ratio of non-positive reviews quickly dropped below the lower bound. This suggests that there may have been successful updates, promotions, or other events that significantly boosted user satisfaction. Studying the positive reviews during these times can help identify the strongest points to maintain and further enhance.